# Second Model: Forecasting GMM

A fundamental limitation of a GMM is that it's a density estimation model. It learns $p(\textbf{x})$ and has no concept of time evolution $p(\textbf{x}_{t+1}|\textbf{x}_t)$ at time-step $t$, like a Hidden Markov Model. Our previous model is good at evaluating whether a current vector of environmental factors and vegetation indices is likely to occur given the ones already observed, but unfortunately it cannot be used in its original form as a forecasting model. We now consider a proposal that the GMM can be modified to allow for forecasting by making use of the EM-algorithm.

Eirola and Lendasse (n.d.) proposed that, making use of **delay embedding** of length $d$ to create $d$-dimensional overlapping rolling windows from a time series, we can fit a GMM to the resulting vectors and forecast future observations using the conditional expectation of a multivariate Gaussian. This allows for the prediction of an entire future horizon simultaneously.

This notebook is a partial implementation of the second section of this paper. Upon successful fitting of this model, we will be able to answer the question: **Will the observed area become stressed in the future**?

## Libraries

In [2]:
import sys
from pathlib import Path
import ee

sys.path.append(str(Path.cwd().parent / "src"))
print("Authenticating with Google Earth Engine...")
ee.Authenticate()
print("Initializing project...")
ee.Initialize(project="agriculture-drought-assesment")

Authenticating with Google Earth Engine...
Initializing project...


In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture 
from sklearn.decomposition import PCA

from data import utils

## Functions

In [4]:
# Extract extreme drought years 2018, 2019

def get_anomaly_df(df):
    # df = rem_year(df, 2025)
    mask = (df["Ag_year"].isin([2018, 2019, 2023, 2024]))
    return df[~mask].copy(), df[mask].copy()

def rem_year(df, year, *years):
    all_years = [year, *years]
    mask = df["Timestamp"].dt.year.isin(all_years)
    return df[~mask]

def get_interpolated_df(spectral, timestamps):
    spectral_columns = [
        "NDVI_mean",
        "NDWI_mean",
        "NDRE_mean",
    ]

    df = spectral.merge(timestamps, on="Timestamp", how="right")
    interpolated_df = df[["Timestamp"] + spectral_columns].copy()
    interpolated_df[spectral_columns] = interpolated_df[spectral_columns].interpolate(method="pchip", limit_area="inside").bfill()

    return interpolated_df.dropna()

## Data

In [5]:
non_spectral_df = pd.read_csv("../data/raw/non_spectral_df.csv", index_col=0)
non_spectral_df["Timestamp"] = pd.to_datetime(non_spectral_df["Timestamp"])

spectral_df = pd.read_csv("../data/raw/spectral_df.csv", index_col=0)
spectral_df["Timestamp"] = pd.to_datetime(spectral_df["Timestamp"])

grouped_spectral_df = pd.read_csv("../data/raw/grouped_spectral_df.csv", index_col=0)
grouped_spectral_df["Timestamp"] = pd.to_datetime(grouped_spectral_df["Timestamp"])

In [6]:
eng_non_spectral_df  = pd.read_csv("../data/processed/eng_non_spectral_df.csv", index_col=0)
eng_non_spectral_df["Timestamp"] = pd.to_datetime(eng_non_spectral_df["Timestamp"])
eng_non_spectral_df["in_season"] = eng_non_spectral_df["in_season"].astype('category')
eng_non_spectral_df["Growth_Stage"] = eng_non_spectral_df["Growth_Stage"].astype('category')

In [7]:
# Rolling statistics and lagged variables
roll_lag_df = pd.read_csv("../data/processed/roll_lag_df.csv", index_col=0)
roll_lag_df["Timestamp"] = pd.to_datetime(roll_lag_df["Timestamp"])

In [8]:
non_spectral_df = non_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)

grouped_spectral_df = grouped_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)

### Get Anomaly Data

In [9]:
norm_ns_df, anom_ns_df = get_anomaly_df(non_spectral_df)
norm_s_df, anom_s_df = get_anomaly_df(grouped_spectral_df[["Timestamp", "Ag_year", "NDVI_mean", "NDWI_mean", "NDRE_mean"]])

norm_eng_ns_df, anom_eng_ns_df = get_anomaly_df(eng_non_spectral_df)
norm_roll_lag_df, anom_roll_lag_df = get_anomaly_df(roll_lag_df)

### Interpolation

The paper by Eirola and Lendasse (n.d.) makes use of a modified EM algorithm that allows for the imputation of missing time series values under the assumption that the observations are missing at random (MAR). We will not use this since vegetation indices follow quite a predictable pattern that can mostly be captured by interpolation (as seen in the previous GMM).

In [10]:
norm_interpolated_df = get_interpolated_df(spectral=norm_s_df, timestamps=norm_eng_ns_df["Timestamp"])
anom_interpolated_df = get_interpolated_df(spectral=anom_s_df, timestamps=anom_eng_ns_df["Timestamp"])

---

## Feature Arrays

In [11]:
eng_features = [
    # Timestamp for merging
    'Timestamp',

    # Date features for plotting
    "shifted_doy",
    "Ag_year",

    # engineered features
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    #'VW_PC1',  
    #'Shallow_mean',
    'ST_PC1'

    # Categoricals not included
    #'Growth_Stage',
    #'in_season',
]

non_spectral_features = [
    'Timestamp',
    'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    'precipitation',
    'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    'soil_temperature_level_4'
]


## Delay Embedding

As previously stated, a GMM operates on vectors rather than sequential observations. Consequently, we transform the time series into _overlapping_ windows of length $d$. If $$\textbf{z}=[z_0,z_1,...,z_{n-1}]$$
and we choose the dimension of embedding as $d$, then _each training sample_ is constructed as $$\textbf{x}_t=[z_t, z_{t+1}, ..., z_{t+d-1}], \quad t=0,1,...,n-d$$
From this we form the new data matrix $X:(n-d+1)\times d$ where the rows are in $\mathbb{R}^d$. 

An important thing to note is that this is for one time series - we consider several related variables. In other words, we embed a multivariate time series, with $m$ variables per each observation $t=0,1,...,n-1$, such that $X\in\mathbb{R}^{(n-d+1)\times(md)}$

In [12]:
def delay_embedding(X, d):
    X = np.asarray(X)

    n, m = X.shape

    embedded = np.empty(shape=(n-d+1, m * d), dtype=X.dtype)

    for i in range(n-d+1):
        embedded[i] = X[i:i+d].reshape(-1)

    return embedded

In [13]:
# Get normal data
X_part1 = norm_eng_ns_df[eng_features]
X_part2 = norm_ns_df[non_spectral_features]
X_non_spectral_norm = pd.merge(X_part2, X_part1, on="Timestamp")

# Merge with interpolated spectral dataframe
X_norm = norm_interpolated_df.merge(X_non_spectral_norm, on="Timestamp", how="left")
X_norm['doy_sin'] = np.sin(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm['doy_cos'] = np.cos(2 * np.pi * X_norm['shifted_doy'] / 365.25)
X_norm["Ag_year"] = X_norm["Ag_year"].astype('int32')
X_norm = X_norm[X_norm["Ag_year"] != 2017]

X_norm.head()

,Timestamp,NDVI_mean,NDWI_mean,NDRE_mean,temperature_2m,precipitation,volumetric_soil_water_layer_3,soil_temperature_level_4,shifted_doy,Ag_year,cumulative_GDD,root_weighted_soil_moisture,ST_PC1,doy_sin,doy_cos
299,2020-10-26,0.259393,-0.073845,0.169762,294.074831,2.856174,0.272553,287.949409,1,2020,12.170304,0.000000,2.035597,0.017202,0.999852
300,2020-10-27,0.262177,-0.072789,0.171346,294.209364,0.471071,0.272286,288.009024,2,2020,23.301050,0.000000,1.901475,0.034398,0.999408
301,2020-10-28,0.265556,-0.071894,0.173272,294.464346,0.789366,0.272016,288.070667,3,2020,34.917067,0.000000,1.909690,0.051584,0.998669
302,2020-10-29,0.269440,-0.071119,0.175494,295.146123,0.084894,0.271727,288.133625,4,2020,47.157650,0.000000,1.941213,0.068755,0.997634
303,2020-10-30,0.273740,-0.070424,0.177960,292.922878,0.906201,0.271408,288.197418,5,2020,57.976044,0.292052,1.931299,0.085906,0.996303


In [14]:
# Get anomaly data

X_part1 = anom_eng_ns_df[eng_features]
X_part2 = anom_ns_df[non_spectral_features]
X_ns_anom = pd.merge(X_part2, X_part1, on="Timestamp")

# Merge with interpolated spectral dataframe
X_anom = anom_interpolated_df.merge(X_ns_anom, on="Timestamp", how="left")
X_anom['doy_sin'] = np.sin(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom['doy_cos'] = np.cos(2 * np.pi * X_anom['shifted_doy'] / 365.25)
X_anom["Ag_year"] = X_anom["Ag_year"].astype('int32')
X_anom = X_anom[X_anom["Ag_year"] != 2017]

X_anom.head()

,Timestamp,NDVI_mean,NDWI_mean,NDRE_mean,temperature_2m,precipitation,volumetric_soil_water_layer_3,soil_temperature_level_4,shifted_doy,Ag_year,cumulative_GDD,root_weighted_soil_moisture,ST_PC1,doy_sin,doy_cos
0,2018-10-27,0.276227,-0.060876,0.18429,294.249324,0.000000,0.275435,287.973715,1,2018,10.490344,0.0,1.627398,0.017202,0.999852
1,2018-10-28,0.276227,-0.060876,0.18429,294.537705,0.000000,0.274899,288.015111,2,2018,21.424214,0.0,1.835885,0.034398,0.999408
2,2018-10-29,0.276227,-0.060876,0.18429,296.162457,0.000000,0.274380,288.061131,3,2018,33.867157,0.0,2.057644,0.051584,0.998669
3,2018-10-30,0.276227,-0.060876,0.18429,296.258737,1.363708,0.273886,288.111546,4,2018,47.485384,0.0,2.143829,0.068755,0.997634
4,2018-10-31,0.276227,-0.060876,0.18429,290.560338,14.212205,0.273621,288.166287,5,2018,54.983462,0.0,1.480230,0.085906,0.996303


In [15]:
scaler = StandardScaler()

X = X_norm.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos"], axis=1).copy()

train_mask = X_norm["Ag_year"] < 2022
test_mask = X_norm["Ag_year"] >= 2022

X_train_raw = X[train_mask]
X_test_raw = X[test_mask]

scaler.fit(X_train_raw)

X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_raw.index)

X_train = pd.concat([X_train, X_norm.loc[train_mask, ["doy_sin", "doy_cos"]]], axis=1)
X_test = pd.concat([X_test, X_norm.loc[test_mask, ["doy_sin", "doy_cos"]]], axis=1)

X_train = delay_embedding(X_train, 24)
X_test = delay_embedding(X_test, 24)

In [16]:
Xa = X_anom.drop(["Timestamp", "shifted_doy", "Ag_year", "doy_sin", "doy_cos"], axis=1).copy()

X_anom_scaled = scaler.transform(Xa)
X_anom_df = pd.DataFrame(X_anom_scaled, columns=Xa.columns, index=X_anom.index)
X_anom_df = pd.concat([X_anom_df, X_anom[["doy_sin", "doy_cos"]]], axis=1)

X_anom_embedded = delay_embedding(X_anom_df, 24)

---

## Conditional Expectations (i.e. Forecasting)

With our regressor size of $d=24$, we can for instance take the last year's measurements as the _first_ 12 months ($P$, known), then calculate the conditional expectation of the next 12 months ($F$, unknown).  The **expectation** step of the EM-algorithm used to fit the GMM on the embedded vectors provides a direct way to find this value. 

# References

Eirola, E. and Lendasse, A. (n.d.). Gaussian Mixture Models for Time Series Modelling, Forecasting, and Interpolation. [online] Available at: https://research.cs.aalto.fi/aml/Publications/Publication204.pdf